In [ ]:
import os
import pandas as pd
import torch
import random
import numpy as np
import torch.nn as nn
from sklearn import preprocessing
import torchvision
import torchaudio  # importing torchaudio library for audio processing
from tqdm import tqdm
import torch.nn.functional as F
import torch.nn.init as init
import torch.optim as optim
import matplotlib.pyplot as plt
from torchvision import transforms
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader, random_split
from types import SimpleNamespace


In [ ]:
cfg = SimpleNamespace()

# paths
cfg.data_folder = ''
cfg.name = "ftt1"
cfg.data_dir = "../input/birdclef-2023/"
cfg.train_data_folder = cfg.data_dir + "train_audio/"
cfg.val_data_folder = cfg.data_dir + "train_audio/"
cfg.test_data_folder = cfg.data_dir + "test_soundscapes/soundscape_29201.ogg"

cfg.birds = np.array(['abethr1', 'abhori1', 'abythr1', 'afbfly1', 'afdfly1', 'afecuc1',
       'affeag1', 'afgfly1', 'afghor1', 'afmdov1', 'afpfly1', 'afpkin1',
       'afpwag1', 'afrgos1', 'afrgrp1', 'afrjac1', 'afrthr1', 'amesun2',
       'augbuz1', 'bagwea1', 'barswa', 'bawhor2', 'bawman1', 'bcbeat1',
       'beasun2', 'bkctch1', 'bkfruw1', 'blacra1', 'blacuc1', 'blakit1',
       'blaplo1', 'blbpuf2', 'blcapa2', 'blfbus1', 'blhgon1', 'blhher1',
       'blksaw1', 'blnmou1', 'blnwea1', 'bltapa1', 'bltbar1', 'bltori1',
       'blwlap1', 'brcale1', 'brcsta1', 'brctch1', 'brcwea1', 'brican1',
       'brobab1', 'broman1', 'brosun1', 'brrwhe3', 'brtcha1', 'brubru1',
       'brwwar1', 'bswdov1', 'btweye2', 'bubwar2', 'butapa1', 'cabgre1',
       'carcha1', 'carwoo1', 'categr', 'ccbeat1', 'chespa1', 'chewea1',
       'chibat1', 'chtapa3', 'chucis1', 'cibwar1', 'cohmar1', 'colsun2',
       'combul2', 'combuz1', 'comsan', 'crefra2', 'crheag1', 'crohor1',
       'darbar1', 'darter3', 'didcuc1', 'dotbar1', 'dutdov1', 'easmog1',
       'eaywag1', 'edcsun3', 'egygoo', 'equaka1', 'eswdov1', 'eubeat1',
       'fatrav1', 'fatwid1', 'fislov1', 'fotdro5', 'gabgos2', 'gargan',
       'gbesta1', 'gnbcam2', 'gnhsun1', 'gobbun1', 'gobsta5', 'gobwea1',
       'golher1', 'grbcam1', 'grccra1', 'grecor', 'greegr', 'grewoo2',
       'grwpyt1', 'gryapa1', 'grywrw1', 'gybfis1', 'gycwar3', 'gyhbus1',
       'gyhkin1', 'gyhneg1', 'gyhspa1', 'gytbar1', 'hadibi1', 'hamerk1',
       'hartur1', 'helgui', 'hipbab1', 'hoopoe', 'huncis1', 'hunsun2',
       'joygre1', 'kerspa2', 'klacuc1', 'kvbsun1', 'laudov1', 'lawgol',
       'lesmaw1', 'lessts1', 'libeat1', 'litegr', 'litswi1', 'litwea1',
       'loceag1', 'lotcor1', 'lotlap1', 'luebus1', 'mabeat1', 'macshr1',
       'malkin1', 'marsto1', 'marsun2', 'mcptit1', 'meypar1', 'moccha1',
       'mouwag1', 'ndcsun2', 'nobfly1', 'norbro1', 'norcro1', 'norfis1',
       'norpuf1', 'nubwoo1', 'pabspa1', 'palfly2', 'palpri1', 'piecro1',
       'piekin1', 'pitwhy', 'purgre2', 'pygbat1', 'quailf1', 'ratcis1',
       'raybar1', 'rbsrob1', 'rebfir2', 'rebhor1', 'reboxp1', 'reccor',
       'reccuc1', 'reedov1', 'refbar2', 'refcro1', 'reftin1', 'refwar2',
       'rehblu1', 'rehwea1', 'reisee2', 'rerswa1', 'rewsta1', 'rindov',
       'rocmar2', 'rostur1', 'ruegls1', 'rufcha2', 'sacibi2', 'sccsun2',
       'scrcha1', 'scthon1', 'shesta1', 'sichor1', 'sincis1', 'slbgre1',
       'slcbou1', 'sltnig1', 'sobfly1', 'somgre1', 'somtit4', 'soucit1',
       'soufis1', 'spemou2', 'spepig1', 'spewea1', 'spfbar1', 'spfwea1',
       'spmthr1', 'spwlap1', 'squher1', 'strher', 'strsee1', 'stusta1',
       'subbus1', 'supsta1', 'tacsun1', 'tafpri1', 'tamdov1', 'thrnig1',
       'trobou1', 'varsun2', 'vibsta2', 'vilwea1', 'vimwea1', 'walsta1',
       'wbgbir1', 'wbrcha2', 'wbswea1', 'wfbeat1', 'whbcan1', 'whbcou1',
       'whbcro2', 'whbtit5', 'whbwea1', 'whbwhe3', 'whcpri2', 'whctur2',
       'wheslf1', 'whhsaw1', 'whihel1', 'whrshr1', 'witswa1', 'wlwwar',
       'wookin1', 'woosan', 'wtbeat1', 'yebapa1', 'yebbar1', 'yebduc1',
       'yebere1', 'yebgre1', 'yebsto1', 'yeccan1', 'yefcan', 'yelbis1',
       'yenspu1', 'yertin1', 'yesbar1', 'yespet1', 'yetgre1', 'yewgre1'])


cfg.n_classes = len(cfg.birds)

cfg.train_df = "../input/birdclef-2023/train_metadata.csv"


In [ ]:
class AudioUtil():
    @staticmethod
    def open(audio_file):
        sig, sr = torchaudio.load(audio_file)
        return (sig, sr)

    @staticmethod
    def rechannel(aud, new_channel):
        sig, sr = aud

        if (sig.shape[0] == new_channel):
          # Nothing to do
          return aud

        if (new_channel == 1):
          # Convert from stereo to mono by selecting only the first channel
          resig = sig[:1, :]
        else:
          # Convert from mono to stereo by duplicating the first channel
          resig = torch.cat([sig, sig, sig])

        return ((resig, sr))

    @staticmethod
    def resample(aud, newsr):
        sig, sr = aud

        if (sr == newsr):
        # Nothing to do
            return aud

        num_channels = sig.shape[0]
        # Resample first channel
        resig = torchaudio.transforms.Resample(sr, newsr)(sig[:1,:])
        if (num_channels > 1):
        # Resample the second channel and merge both channels
            retwo = torchaudio.transforms.Resample(sr, newsr)(sig[1:,:])
            resig = torch.cat([resig, retwo])

        return ((resig, newsr))

    @staticmethod
    def pad_trunc(aud, max_ms):
        sig, sr = aud
        num_rows, sig_len = sig.shape
        max_len = sr//1000 * max_ms

        if (sig_len > max_len):
        # Truncate the signal to the given length
            sig = sig[:,:max_len]

        elif (sig_len < max_len):
            # Length of padding to add at the beginning and end of the signal
            pad_begin_len = random.randint(0, max_len - sig_len)
            pad_end_len = max_len - sig_len - pad_begin_len

            # Pad with 0s
            pad_begin = torch.zeros((num_rows, pad_begin_len))
            pad_end = torch.zeros((num_rows, pad_end_len))

            sig = torch.cat((pad_begin, sig, pad_end), 1)

        return (sig, sr)

    @staticmethod
    def time_shift(aud, shift_limit):
        sig,sr = aud
        _, sig_len = sig.shape
        shift_amt = int(random.random() * shift_limit * sig_len)
        return (sig.roll(shift_amt), sr)

    @staticmethod
    def spectro_gram(aud, n_mels=64, n_fft=1024, hop_len=None):
        sig,sr = aud
        top_db = 80

        # spec has shape [channel, n_mels, time], where channel is mono, stereo etc
        spec = torchaudio.transforms.MelSpectrogram(sr, n_fft=n_fft, hop_length=hop_len, n_mels=n_mels)(sig)

        # Convert to decibels
        spec = torchaudio.transforms.AmplitudeToDB(top_db=top_db)(spec)
        return (spec)

    @staticmethod
    def spectro_augment(spec, max_mask_pct=0.1, n_freq_masks=1, n_time_masks=1):
        _, n_mels, n_steps = spec.shape
        mask_value = spec.mean()
        aug_spec = spec

        freq_mask_param = max_mask_pct * n_mels
        for _ in range(n_freq_masks):
            aug_spec = torchaudio.transforms.FrequencyMasking(freq_mask_param)(aug_spec, mask_value)

        time_mask_param = max_mask_pct * n_steps
        for _ in range(n_time_masks):
            aug_spec = torchaudio.transforms.TimeMasking(time_mask_param)(aug_spec, mask_value)

        return aug_spec

In [ ]:
def preprocessData(aud):
    duration = 8000
    sr = 32000
    channel = 3
    shift_pct = 0.4
    reaud = AudioUtil.resample(aud, sr)
    rechan = AudioUtil.rechannel(reaud, channel)
    dur_aud = AudioUtil.pad_trunc(rechan, duration)
    shift_aud = AudioUtil.time_shift(dur_aud, shift_pct)
    sgram = AudioUtil.spectro_gram(shift_aud, n_mels=64, n_fft=1024, hop_len=None)
    aug_sgram = AudioUtil.spectro_augment(sgram, max_mask_pct=0.1, n_freq_masks=2, n_time_masks=2)
    aug_sgram_m, aug_sgram_s = aug_sgram.mean(), aug_sgram.std()
    aug_sgram = (aug_sgram - aug_sgram_m) / aug_sgram_s
    return aug_sgram

In [ ]:
# 定义数据类处理文件
class RawData:

    __train_data_path = cfg.train_data_folder
    
    __train_df = pd.read_csv(cfg.train_df)
    
    __labels_t = None
    __audio_names = None
    __le=preprocessing.LabelEncoder()

    @staticmethod
    def labels_t():
        if RawData.__labels_t is None:
            labels = RawData.__train_df['primary_label']
            #RawData.__le = preprocessing.LabelEncoder()
            RawData.__labels_t = RawData.__le.fit_transform(labels)
            
            #RawData.__labels_t = torch.as_tensor(targets)
        return RawData.__labels_t

    @staticmethod
    def audio_names():
        if RawData.__audio_names is None:
            RawData.__audio_names = RawData.__train_df['filename']
        return RawData.__audio_names

    @staticmethod
    def re_transf(data):
        return RawData.__le.inverse_transform(data)

In [ ]:
def MakeFrame(audio, duration=5, sr=32000):
    frame_length = int(duration * sr)
    frame_step = int(duration * sr)
    if frame_length>audio.shape[1]:
        return audio.unsqueeze(0)
        #audio = torch.tensor(audio)
    else:
        chunks = audio.unfold(1, frame_length, frame_step).transpose(0, 1)
        return chunks

In [ ]:
import shutil
#data_path = '/kaggle/working/'
save_data_path='/kaggle/working/data/'
if os.path.exists(save_data_path):
    shutil.rmtree(save_data_path)
t_data_path=save_data_path+'data/'
os.mkdir(save_data_path) 
os.mkdir(t_data_path) 
for tit in cfg.birds[:132]:
    os.mkdir(t_data_path+tit) 

In [ ]:
cfg.birds[132]

In [ ]:
train_df = pd.read_csv(cfg.train_df)
n=len(train_df)
#k=int(n/2)

In [ ]:
lists = []
for i in range(n):
    print(i,end='\r')
    data = train_df.iloc[i]
    if data['primary_label']=='lesmaw1':
        print(i)
        break
    #print(data['primary_label'])
    loadpath = cfg.train_data_folder+ data['filename']
    aud,sr = torchaudio.load(loadpath)  
    #print(aud.shape)
    signal=preprocessData((aud,sr))
    savepath = t_data_path+ data['filename'][:-4]+'.pth'
    torch.save(signal,savepath)
    lists.append([data['primary_label'],data['filename']])

    #break

In [ ]:
'''train_df = pd.read_csv(cfg.train_df)
lists = []
for i in range(len(train_df)):
    data = train_df.iloc[i]
    #print(data['primary_label'])
    loadpath = cfg.train_data_folder+ data['filename']
    aud,sr = torchaudio.load(loadpath)  
    #print(aud.shape)
    frames=MakeFrame(aud)
    #print(frames.shape)
    i=5
    for frame in frames:
        signal=preprocessData((frame,sr))
        savepath = save_data_path+ data['filename'][:-4]+'_%d'%i+'.pth'
        torch.save(signal,savepath)
        lists.append([data['primary_label'],savepath])
        i+=5
    #break'''

In [ ]:
pd_lists=pd.DataFrame(lists)
pd_lists.to_csv(save_data_path+'lists.csv',index=False)

In [ ]:
import os
import zipfile
import datetime

def file2zip(packagePath, zipPath):
    '''
  :param packagePath: 文件夹路径
  :param zipPath: 压缩包路径
  :return:
  '''
    zip = zipfile.ZipFile(zipPath, 'w', zipfile.ZIP_DEFLATED)
    for path, dirNames, fileNames in os.walk(packagePath):
        fpath = path.replace(packagePath, '')
        for name in fileNames:
            fullName = os.path.join(path, name)
            name = fpath + '\\' + name
            zip.write(fullName, name)
    zip.close()



# 文件夹路径
#packagePath = '/kaggle/working/'
zipPath = save_data_path+'output.zip'
if os.path.exists(zipPath):
    os.remove(zipPath)
file2zip(t_data_path, zipPath)
print("打包完成")
print(datetime.datetime.utcnow())
